In [ ]:
!pip install torch torchvision transformers pillow gradio requests -q

In [ ]:
import os
import requests
import torch
from PIL import Image
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration

# 🔴 Add your Groq API Key
GROQ_API_KEY = "your_groq_api_key_here"

MODEL_NAME = "llama-3.1-8b-instant"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

In [ ]:
print("Loading BLIP model...")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

model.to(DEVICE)

print("✅ BLIP loaded successfully!")

In [ ]:
def generate_caption(image):
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=60)

    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption

In [ ]:
def query_llama(prompt):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a precise vision assistant."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.2,
        "max_tokens": 300
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code != 200:
        return f"❌ API Error: {response.text}"

    return response.json()["choices"][0]["message"]["content"]

In [ ]:
def vision_rag_pipeline(image, question):
    if image is None:
        return "No image", "Upload an image", "❌"

    image = Image.fromarray(image)

    caption = generate_caption(image)

    prompt = f"""
IMAGE DESCRIPTION:
{caption}

QUESTION:
{question}

Answer strictly from the image.
"""

    answer = query_llama(prompt)

    return caption, answer, "✅ Done"

In [ ]:
def build_ui():
    with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as app:

        gr.Markdown("""
        # 🧠 Vision RAG Assistant
        Upload image and ask questions
        """)

        with gr.Row():
            with gr.Column():
                image_input = gr.Image(type="numpy", label="Upload Image")
                question_input = gr.Textbox(label="Question")
                btn = gr.Button("Analyze", variant="primary")

            with gr.Column():
                caption = gr.Textbox(label="Caption")
                answer = gr.Textbox(label="Answer")
                status = gr.Textbox(label="Status")

        btn.click(
            fn=vision_rag_pipeline,
            inputs=[image_input, question_input],
            outputs=[caption, answer, status]
        )

    return app

In [ ]:
app = build_ui()
app.launch(debug=True)